In [12]:
import pandas as pd
import numpy as np

In [13]:
df = pd.read_csv(r"C:\Users\Sairaj\OneDrive\Desktop\Movie Recommendation Project\Datasets\movies.csv")

In [14]:
df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

df['genres'] = df['genres'].str.replace('|', ' ', regex = False)


df['tdidf'] = df['title'] + " " + df['genres']


print(df['tdidf'].head())


tfidf = TfidfVectorizer(stop_words = 'english')


tfidf_matrix = tfidf.fit_transform(df['tdidf'])

0    Toy Story (1995) Adventure Animation Children ...
1            Jumanji (1995) Adventure Children Fantasy
2               Grumpier Old Men (1995) Comedy Romance
3        Waiting to Exhale (1995) Comedy Drama Romance
4            Father of the Bride Part II (1995) Comedy
Name: tdidf, dtype: object


In [16]:
import pickle 

with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(tfidf, f)
    
with open('tfidf_matrix.pkl', 'wb') as f:
    pickle.dump(tfidf_matrix, f)

In [19]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def icebreaker(movie_list, n_recs = 100):
    movie_indices = []
    
    for title in movie_list:
        matches = df[df['title'].str.contains(title, case = False, na = False)]
        
        if len(matches) > 0:
            best_match_idx = matches['title'].str.len().idxmin()
            
            movie_indices.append(best_match_idx)
            print(f"Found: {df.loc[best_match_idx, 'title']}")
        else:
            print(f"Could not find: {title}")


    target_vectors = tfidf_matrix[movie_indices]
    
   
    user_taste_vector = np.asarray(target_vectors.mean(axis = 0))
    
    
    similarity_scores = cosine_similarity(user_taste_vector, tfidf_matrix).flatten()
    
    
    sorted_indices = similarity_scores.argsort()[::-1]
    
    
    top_indices = [i for i in sorted_indices if i not in movie_indices]
    
    
    final_indices = top_indices[:n_recs]
    

    results = df.iloc[final_indices][['movieId', 'title', 'genres']].copy()
    results['Match Score (%)'] = (similarity_scores[final_indices] * 100).round(2)
    
    return results


my_top_3 = ["Matrix", "Superbad", "Toy Story"]

recommendations = icebreaker(my_top_3, n_recs = 100)

print("\n TOP 10 RECOMMENDATIONS (Out of 100):")
recommendations.head(10)

Found: Matrix, The (1999)
Found: Superbad (2007)
Found: Toy Story (1995)

 TOP 10 RECOMMENDATIONS (Out of 100):


,movieId,title,genres,Match Score (%)
3021,3114,Toy Story 2 (1999),Adventure Animation Children Comedy Fantasy,59.47
46005,171559,Superbad (2016),Comedy,51.01
59767,201588,Toy Story 4 (2019),Adventure Animation Children Comedy,48.57
14813,78499,Toy Story 3 (2010),Adventure Animation Children Comedy Fantasy IMAX,46.83
20497,106022,Toy Story of Terror (2013),Animation Children Comedy,40.96
25157,123109,P.U.N.K.S (1999),Children Comedy Sci-Fi,40.55
11423,51412,Next (2007),Action Sci-Fi Thriller,37.46
6247,6365,"Matrix Reloaded, The (2003)",Action Adventure Sci-Fi Thriller IMAX,36.87
6809,6934,"Matrix Revolutions, The (2003)",Action Adventure Sci-Fi Thriller IMAX,36.54
2869,2961,"Story of Us, The (1999)",Comedy Drama,35.66
